### These are live databases

Re-running retrieval today will not return the study's exact intermediate tables. Open
Targets re-scores associations every release, CIViC gains curated evidence continuously
and STRING accumulates interaction evidence. Treat the stored tables as the record of
what the paper used, and this notebook as how a reader re-derives the pipeline or
refreshes it when a referee asks for a current snapshot.

A full run of this notebook was verified against the published values:

| | rebuilt from live APIs | published |
|---|---|---|
| seed genes | 186 | 186 |
| genes shared with the published seed set | 184 of 186 | — |
| network nodes | 186 | 186 |
| network edges | 1,564 | 1,421 |
| giant component | 176 | 176 |
| isolated seeds | 10 | 10 |
| modules detected | 5 | 4 |
| **target panel** | **EGFR, KDR, PIK3CA, PDGFRA, CDK6, IDH1** | **identical** |

The edge count is higher because STRING has gained evidence since the study ran, and the
denser graph splits one module in two. Neither moves the panel: selection depends on
centrality and per-hub ChEMBL compound counts, not on the module partition, and all six
published targets are recovered.

## 0. Setup

Works on a laptop, Colab or Kaggle. Retrieval is network-bound, not compute-bound: no
GPU helps any step here.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path.cwd()
assert (REPO / "repro" / "fetch_sources.py").exists(), (
    "run this notebook from the root of the extracted module "
    "(the directory containing repro/, stages/ and config/)")
sys.path.insert(0, str(REPO))

# Where retrieved inputs and outputs live. On Colab or Kaggle point this at storage
# that outlives the session, or a dropped runtime takes the ChEMBL download with it.
os.environ.setdefault("MARINE_REPRO_DATA", str(REPO / "data" / "retrieved"))
Path(os.environ["MARINE_REPRO_DATA"]).mkdir(parents=True, exist_ok=True)
WORK = REPO / "work_stage01"; WORK.mkdir(exist_ok=True)

# The stage scripts import repro.config, so the repo root must be on PYTHONPATH for
# any subprocess - this is what run_stage.sh sets, and omitting it fails at step 08.
ENV = dict(os.environ, PYTHONPATH=str(REPO))

print("module root:", REPO)
print("data dir   :", os.environ["MARINE_REPRO_DATA"])

In [ ]:
# One reachability check per service, before committing to a long run.
import urllib.request, urllib.error
for name, url in [
    ("Open Targets", "https://api.platform.opentargets.org/api/v4/graphql"),
    ("CIViC",        "https://civicdb.org/api/graphql"),
    ("STRING",       "https://string-db.org/api/json/version"),
    ("Reactome",     "https://reactome.org/AnalysisService/database/version"),
    ("ChEMBL",       "https://www.ebi.ac.uk/chembl/api/data/status?format=json"),
    ("ZINC",         "https://files.docking.org"),
]:
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=25)
        print(f"  {name:13s} reachable")
    except urllib.error.HTTPError as e:
        print(f"  {name:13s} reachable (HTTP {e.code} to a HEAD request is fine)")
    except Exception as e:
        print(f"  {name:13s} UNREACHABLE - {type(e).__name__}: {str(e)[:60]}")

## 1. Stage 01 — the glioma network and the target panel

### 1a. Open Targets and CIViC

Six MONDO terms are queried for disease-to-target associations, and CIViC for accepted
clinical evidence. CIViC supplies genes that carry curated glioma evidence but sit below
the association-score cut, which is why three of them enter the seed set on evidence
rather than score.

In [ ]:
from repro.fetch_sources import (fetch_open_targets, fetch_civic, fetch_string,
                                  fetch_reactome, GLIOMA_DISEASES, CIVIC_DISEASES)

print("Open Targets:")
ot = fetch_open_targets(GLIOMA_DISEASES, n=200)
print("\nCIViC:")
civic = fetch_civic(CIVIC_DISEASES)

print(f"\nretrieved {sum(len(b['rows']) for b in ot.values())} association rows "
      f"across {len(ot)} disease terms")
print(f"retrieved {sum(b['total_count'] for b in civic.values())} accepted evidence items")

### 1b. Seed-gene selection and the STRING network

The study's own script does the selection. It runs in two passes because it derives the
symbol list itself and only then asks for the network: the first pass stops at the
STRING call, we fetch the network for the symbols it produced, and the second pass
completes. The shim raises rather than returning an empty network, so a missing payload
cannot silently shrink the seed set.

In [ ]:
from repro.host_shim import HostShim, run_legacy

SCRIPT = str(REPO / "stages" / "legacy" / "stage01" / "03_fetch_string_network.py")

# pass 1 - runs the selection logic, stops at the STRING call
try:
    run_legacy(SCRIPT, HostShim(open_targets=ot, civic=civic), workdir=str(WORK))
except RuntimeError as e:
    assert "no STRING payload" in str(e), e
    print("pass 1 complete - selection logic ran, STRING call reached")

symbols = json.load(open(WORK / "handoff" / "seed_symbols.json"))
print(f"seed genes selected by the study's own code: {len(symbols)}")

# pass 2 - fetch the network for exactly those symbols, then complete the script
net = fetch_string(symbols, species=9606, required_score=700)
run_legacy(SCRIPT, HostShim(open_targets=ot, civic=civic, string=net), workdir=str(WORK))
print("\npass 2 complete")

In [ ]:
# Topology, against the published network
import networkx as nx, pandas as pd

G = nx.Graph()
for n in net["nodes"]:
    G.add_node(n["name"])
for e in net["edges"]:
    G.add_edge(e["a"], e["b"], weight=e["score"])
giant = max(nx.connected_components(G), key=len)

seed = pd.read_csv(WORK / "glioma_seed_genes.csv")
published = pd.read_csv(REPO / "data" / "shipped" / "glioma_seed_genes.csv")
shared = set(seed.gene_symbol) & set(published.gene_symbol)

print(f"{'':22s} {'rebuilt':>10s} {'published':>10s}")
print(f"{'seed genes':22s} {len(seed):>10d} {len(published):>10d}")
print(f"{'network nodes':22s} {G.number_of_nodes():>10d} {186:>10d}")
print(f"{'network edges':22s} {G.number_of_edges():>10d} {1421:>10d}")
print(f"{'giant component':22s} {len(giant):>10d} {176:>10d}")
print(f"{'isolated seeds':22s} {len(net['isolated']):>10d} {10:>10d}")
print(f"\ngenes shared with the published seed set: {len(shared)} of {len(published)}")
print("edge count above the published value reflects STRING evidence added since the "
      "study ran, not a defect; the topology that the panel selection depends on "
      "- giant component and isolate count - is unchanged.")

### 1c. Modules, centrality and the target panel

These steps are deterministic or already speak HTTP, so they run as shipped. Step 07
queries ChEMBL for the per-hub compound counts that gate the panel.

One thing worth knowing before the next section: **the panel does not depend on the
pathway enrichment.** `08_select_target_panel.py` reads only `chembl_counts.json` and
`centrality.csv`. The Reactome and GO section below is therefore optional — it produces
a descriptive table for the paper, and skipping it costs that table and nothing else.

In [ ]:
# Greedy modularity on the network. Writes handoff/modules.json as {gene: module_label}.
run_legacy(str(REPO / "stages/legacy/stage01/04_detect_modules.py"),
           HostShim(open_targets=ot, civic=civic, string=net), workdir=str(WORK))

mod_of = json.load(open(WORK / "handoff" / "modules.json"))
mod_genes = {}
for gene, m in mod_of.items():
    mod_genes.setdefault(str(m), []).append(gene)
print("modules:", {k: len(v) for k, v in sorted(mod_genes.items())})
print("published: 4 modules of 64/56/43/8 genes, Q = 0.380, plus 15 unassigned")

In [ ]:
# Centrality first, then the ChEMBL target map that step 07 needs.
r = subprocess.run([sys.executable, str(REPO / "stages/legacy/stage01/05_build_centrality_table.py")],
                   cwd=str(WORK), env=ENV, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-400:]
cen = pd.read_csv(WORK / "handoff" / "centrality.csv")
print(f"centrality table: {len(cen)} genes")

# handoff/chembl_targets.json is read by step 07 but written by no script in the module
# - it was produced outside the shipped pipeline in the original run. Build it here.
from repro.fetch_sources import fetch_chembl_targets
hubs = cen.gene_symbol.tolist()
tmap = fetch_chembl_targets(hubs)
(WORK / "handoff").mkdir(exist_ok=True)
json.dump(tmap, open(WORK / "handoff" / "chembl_targets.json", "w"))

# Step 07 then counts compounds per hub, and 08 applies the data threshold. ~10 min.
for step in ["07_fetch_chembl_hub_counts.py", "08_select_target_panel.py",
             "09_export_centrality_table.py"]:
    print(f"-- {step}")
    r = subprocess.run([sys.executable, str(REPO / "stages/legacy/stage01" / step)],
                       cwd=str(WORK), env=ENV, capture_output=True, text=True)
    print((r.stdout or r.stderr)[-400:])
    if r.returncode != 0:
        raise SystemExit(f"{step} failed")

panel = pd.read_csv(WORK / "glioma_target_panel.csv")
print("\nselected panel :", panel.iloc[:, 0].tolist())
print("published panel: ['EGFR', 'KDR', 'PIK3CA', 'PDGFRA', 'CDK6', 'IDH1']")

### 1d. Pathway enrichment — optional

Reactome and GO feed `06_pathway_enrichment.py`, which writes the descriptive pathway
table. Neither is an input to target selection, so this whole subsection is skippable.

**Reactome may refuse this client.** reactome.org sits behind Cloudflare and answers
some automated clients with error 1010 whatever the request looks like; it does so from
the environment this module was prepared in. Do not disguise the client to get around
it. Either run this cell from a network Reactome accepts — Colab often is one — or paste
each module's gene list into the Reactome analysis page and save the JSON yourself. The
function raises with that instruction attached rather than returning an empty result.

GO goes through UniProt and QuickGO and is reliable: QuickGO is keyed on gene-product
accessions rather than symbols, so each symbol is resolved through UniProt first.

In [ ]:
from repro.fetch_sources import fetch_go
DATA = Path(os.environ["MARINE_REPRO_DATA"])

# GO first - this one is dependable
go = fetch_go(sorted(mod_of))
json.dump(go, open(DATA / "go_raw.json", "w"))
print("go_raw.json written:", (DATA / "go_raw.json").stat().st_size / 1e6, "MB")

# Reactome - may raise on this network, which is not fatal
try:
    react = fetch_reactome(mod_genes)
    json.dump(react, open(DATA / "reactome_raw.json", "w"))
    print("reactome_raw.json written, Reactome version",
          next(iter(react.values()))["reactome_version"])
    have_reactome = True
except RuntimeError as e:
    print("Reactome unavailable from this network:\n ", str(e)[:260])
    have_reactome = False

In [ ]:
if have_reactome:
    r = subprocess.run([sys.executable, str(REPO / "stages/legacy/stage01/06_pathway_enrichment.py")],
                       cwd=str(WORK), env=ENV, capture_output=True, text=True)
    print((r.stdout or r.stderr)[-400:])
    enr = pd.read_csv(WORK / "glioma_pathway_enrichment.csv")
    print(f"\nenrichment table: {len(enr)} rows at FDR < 0.05 with >= 3 entities found")
else:
    print("skipped - the pathway table is the only output affected, and the six-target "
          "panel above was selected without it")

## 2. Stage 03 — the ChEMBL label matrix

Already portable; every step calls `https://www.ebi.ac.uk/chembl/api/data` directly.
Budget about **three hours**, nearly all of it waiting on the API rather than computing.
The two censored-data steps are the slow ones.

Run this on storage that survives a dropped session if you are on Colab or Kaggle.

In [ ]:
# ~3 h. Resumable in the sense that each step writes its own output and the runner
# skips a step whose outputs already exist.
!./run_stage.sh 03 --check

In [ ]:
!./run_stage.sh 03

## 3. Stage 04 — control sets

ZINC20 property-matched decoys as negative controls, and the FDA-approved drug-target
pairs as positive controls. The ZINC pool download is the long part, roughly 40 minutes.

In [ ]:
!./run_stage.sh 04 --check

In [ ]:
!./run_stage.sh 04

## 4. Verify, and collect what to bring home

Stage 12 re-asserts the published headline numbers against whatever is now in the data
directory. Run it after retrieval: it is the check that catches a refreshed database
having moved something you relied on.

In [ ]:
!./run_stage.sh 12

In [ ]:
# If you ran this on Colab or Kaggle, these are the files to copy back to the laptop.
import shutil
OUT = Path(os.environ["MARINE_REPRO_DATA"])
produced = ["glioma_seed_genes.csv", "glioma_network_centrality.csv", "glioma_target_panel.csv",
            "glioma_pathway_enrichment.csv", "glioma_ppi_edges.csv", "reactome_raw.json",
            "go_raw.json", "chembl_multitask_matrix.parquet", "chembl_fp.npz",
            "decoy_set.parquet", "fda_positive_controls.csv"]
for f in produced:
    # stage-01 scripts write some outputs to the working directory and some to handoff/
    for src in (WORK / f, WORK / "handoff" / f, OUT / f):
        if src.exists():
            if src.parent != OUT:
                shutil.copy(src, OUT / f)
            print(f"  {f:38s} {src.stat().st_size/1e6:8.2f} MB")
            break
    else:
        print(f"  {f:38s} not produced by the stages run above")
print(f"\ncopy the contents of {OUT} into your laptop's MARINE_REPRO_DATA, then render "
      "figures locally with ./run_stage.sh 11")

## What this notebook does not cover

- **Stage 02, the marine library.** CMNPD has no public API. The curated library and the
  glioma cell-line activity records ship as stored data; stage 02 reads the two
  non-redistributable CMNPD vendor files from your data directory if you want to
  re-derive the curation.
- **Stages 05-10.** Training, evaluation, attribution, screening and the two control
  experiments. None of them retrieve anything; see `GUIDE_recomputing_figures.md` for
  which of those you need and where to run them.